# Семинар 4. Витрина клиента

Сегодня мы соберём одну таблицу, в которой **одна строка — один клиент**, и рядом
с ним всё, что мы про него знаем: сколько подписок, сколько заплатил, пользуется ли
чужой подпиской, из какого канала пришёл.

Такая таблица называется **витриной**. Из неё потом считается всё остальное:
отток, LTV, сегменты, воронка. Если в витрине лишние деньги — они будут лишними
везде.

> **Одна строка — один клиент.** Это и есть **зерно** витрины. Слово с лекции,
> и сегодня оно станет рабочим инструментом: зерно объявляют **словами и до того**,
> как написан первый `merge`.


---

## 1. Каркас: клиенты, подписки, платежи

Первое правило сборки витрины: **контрольное число считается до сборки, а не
после.** Мы заранее знаем, сколько строк обязано получиться, и любое другое число
— это ошибка, а не «ну вот столько вышло».


In [ ]:
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


SEGMENT = "region_code <= 20"          # двадцать регионов из восьмидесяти девяти
WINDOW = ("2026-01-01", "2026-07-01")  # окно фактов: полугодие, правый конец не включён

print("подключились как:", q("select current_user as login").iloc[0, 0])


In [ ]:
clients = q(f"""
    select client_id, region_code, acquisition_channel
    from prime.clients
    where {SEGMENT}
    order by client_id
""")

N = len(clients)
print("ЗЕРНО: одна строка — один клиент.")
print("В витрине обязано получиться строк:", N)


In [ ]:
subs = q(f"""
    select s.subscription_id, s.owner_client_id, s.tariff_id, s.started_at, s.ended_at
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where c.{SEGMENT}
    order by s.subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join prime.clients c on c.client_id = s.owner_client_id
    where c.{SEGMENT}
      and p.status = 'success'
      and p.paid_at >= :a and p.paid_at < :b
    order by p.payment_id
""", a=WINDOW[0], b=WINDOW[1])

print(f"подписок: {len(subs):>8}")
print(f"платежей: {len(pay):>8}")


In [ ]:
# Клиентов 112 415, подписок 75 183. Что должно случиться с клиентом,
# у которого подписки нет: исчезнуть из витрины или остаться с пустотой?
frame = clients.merge(
    subs,
    left_on="client_id",
    right_on="owner_client_id",
    how=ЗАПОЛНИТЕ,
)

print("строк после соединения с подписками:", len(frame), " ждали:", N)


In [ ]:
# А теперь присоединим платежи тем же движением — «просто добавим ещё одну таблицу».
naive = frame.merge(pay, on="subscription_id", how="left")

print(f"строк стало: {len(naive):>8}")
print(f"а клиентов:  {N:>8}")
print(f"раздулось в {len(naive) / N:.1f} раза")


**Соединение не склеивает — оно перемножает.** У клиента шесть платежей за
полугодие, и клиент превращается в шесть строк. Витрины больше нет: одна строка
перестала быть одним клиентом.

Дальше — правило, по которому собираются все витрины на свете. Три шага, и порядок
в них важнее самих шагов:

1. **Объявите зерно результата словами.** «Одна строка — один клиент». До кода.
2. **Сверните сторону «многие» до этого зерна.** `groupby` + `agg`, отдельной
   таблицей.
3. **Соединяйте только то, что уже одного зерна**, и объявляйте кратность
   в `validate=`.

Если третий шаг падает — значит, ошиблись вы на втором, и узнаёте вы это сразу,
а не через неделю от финансов.


In [ ]:
def step(name: str, before: pd.DataFrame, after: pd.DataFrame, key: str,
         validate: str = "", expect_lost: int | None = None) -> dict:
    """Одна строка протокола сборки.

    keys_lost — строки слева, которым не нашлось пары справа.
    Видно их по пропуску в колонке key, пришедшей из правой таблицы.
    """
    lost = int(after[key].isna().sum())
    if len(after) != len(before):
        verdict = f"ВНИМАНИЕ: строк было {len(before)}, стало {len(after)}"
    elif expect_lost is None:
        verdict = "ok"
    elif lost != expect_lost:
        verdict = f"ВНИМАНИЕ: потеряно {lost}, ждали {expect_lost}"
    else:
        verdict = f"ok, ожидали {expect_lost}"
    return {"step": name, "rows_before": len(before), "rows_after": len(after),
            "validate": validate, "keys_lost": lost, "verdict": verdict}


protocol: list[dict] = []
protocol.append(step("клиент -> подписка", clients, frame, "subscription_id",
                     "1:1", N - len(subs)))
print("протокол заведён, первая строка записана")


In [ ]:
# Сворачиваем платежи до клиента ДО соединения.
by_client = (pay
             .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id")
             .groupby("owner_client_id", as_index=False)
             .agg(n_payments=("payment_id", "size"), paid=("amount", "sum")))

# Сколько строк слева и сколько справа встретятся на одном client_id?
# Объявите кратность ДО того, как нажмёте.
mart = frame.merge(
    by_client, left_on="client_id", right_on="owner_client_id",
    how="left", validate=ЗАПОЛНИТЕ, suffixes=("", "_pay"),
)

protocol.append(step("платежи -> клиент", frame, mart, "n_payments", "1:1", N - len(by_client)))
print(len(mart), "строк, ждали", N)


In [ ]:
# Деньги приехали float64 (это мы видели на лекции). Для сверки до копейки
# переводим в целые копейки: сравнивать целое с целым надёжнее.
mart["paid_kopecks"] = (mart["paid"].fillna(0) * 100).round().astype(ЗАПОЛНИТЕ)

print(mart["paid_kopecks"].sum(), "копеек всего")


---

## 2. Справочник цен: ломается не там, где вы думаете

В `prime.tariffs` лежит история цен: у каждого тарифа несколько строк с интервалом
действия. Чтобы узнать, сколько **должен** был заплатить клиент, платёж надо
соединить со справочником **по попаданию в интервал**.

Сюрприз сегодня другой, чем на С1. Дефект найти легко. А вот починить его можно
двумя способами, оба выглядят разумно, и **глазами между ними не выбрать**.


In [ ]:
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

tariffs


In [ ]:
# Пересечения ищем кодом, а не глазами: глазами вы их найдёте на пяти строках
# и не найдёте на пятистах.
a = tariffs.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))

overlap = pairs[
    (pairs["row_id_a"] < pairs["row_id_b"])              # без этого условия каждая
    & (pairs["valid_from_a"] <= pairs["valid_to_b"])      # строка пересечётся сама
    & (pairs["valid_from_b"] <= pairs["valid_to_a"])      # с собой
]

print("пересекающихся пар:", len(overlap))
overlap[["tariff_id", "valid_from_a", "valid_to_a", "valid_from_b", "valid_to_b"]]


**Строка с ценой 429 ₽ перекрывает сразу две соседние.** Она действует
с 1 февраля по 30 июня 2026, а в это же окно попадают и старая цена 399 (до
28 февраля), и новая 449 (с 1 марта).

Значит, каждый платёж февраля-июня найдёт в справочнике **две** подходящие строки
вместо одной, и после соединения платежей станет больше, чем их есть.

Вопрос, на который нет очевидного ответа: **что с этим делать?**

- Выбросить строку 429 — она выглядит как ошибка ввода.
- Оставить её и брать самую свежую подходящую цену — может, цену и правда меняли
  дважды.

Оба варианта разумны. Оба дают правдоподобную выручку. Выбрать между ними глазами
нельзя — и именно поэтому существует **контрольная сумма**.


In [ ]:
# Пробуем соединить по интервалу. Эта ячейка УПАДЁТ — так и задумано.
probe = pay.head(1000).merge(tariffs, how="cross")
probe["valid_from"] = pd.to_datetime(probe["valid_from"])   # даты справочника, без зоны

try:
    hit = probe["paid_at"] >= probe["valid_from"]
    print("сравнилось, строк:", hit.sum())
except TypeError as e:
    print("TypeError:", e)
    print()
    print("Читаем трейсбек СНИЗУ: последняя строка называет, что произошло.")
    print("Слева момент с часовым поясом, справа наивная дата. pandas отказывается")
    print("гадать, в какой зоне записан справочник, — и правильно делает.")


In [ ]:
# В какой зоне лежат даты справочника — и почему это решаем мы, а не pandas?
ref = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize(ЗАПОЛНИТЕ)

print(ref[["tariff_id", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))


In [ ]:
# valid_to — это дата, то есть полночь. Платёж 28 февраля в 14:30 попадает
# в интервал, который кончается 28 февраля?
def join_prices(pay: pd.DataFrame, ref: pd.DataFrame, between: bool) -> pd.DataFrame:
    m = pay.merge(ref[["tariff_id", "monthly_fee", "valid_from", "valid_to"]], on="tariff_id")
    if between:                                  # буквальное BETWEEN
        hit = (m["paid_at"] >= m["valid_from"]) & (m["paid_at"] <= m["valid_to"])
    else:                                        # конвенция курса
        hit = (m["paid_at"] >= m["valid_from"]) & (m["paid_at"] < m["valid_to"] + pd.Timedelta(days=ЗАПОЛНИТЕ))
    return m[hit]


pay_t = pay.merge(subs[["subscription_id", "tariff_id"]], on="subscription_id")
print("платежей с тарифом:", len(pay_t))

# Сразу дёргаем обе ветки на сотне строк: пропуск внутри функции иначе
# не даст о себе знать до самого конца.
print("проба BETWEEN: ", len(join_prices(pay_t.head(100), ref, between=True)))
print("проба конвенции:", len(join_prices(pay_t.head(100), ref, between=False)))


In [ ]:
# Независимый контроль: сумма платежей, посчитанная БЕЗ всякого справочника.
control = pay["amount"].sum()
print(f"контроль: {control:>16,.2f} ₽   ({len(pay)} платежей)".replace(",", " "))
print()

fixed = ref[ref["monthly_fee"] != 429]

rows = []
for ref_name, ref_table in (("как есть", ref), ("без 429", fixed)):
    for between, boundary in ((True, "BETWEEN"), (False, "конвенция курса")):
        res = join_prices(pay_t, ref_table, between)
        rows.append({"reference": ref_name, "boundary": boundary,
                     "rows": len(res), "total": res["amount"].sum(),
                     "diff_vs_control": res["amount"].sum() - control})

pd.DataFrame(rows)


### Ровно одна комбинация из четырёх даёт ноль

| Справочник | Граница | Строк | Разница с контролем |
|---|---|---:|---:|
| как есть | BETWEEN | 308 848 | +32 318 678 ₽ |
| как есть | конвенция курса | 310 688 | +32 915 038 ₽ |
| без 429 | BETWEEN | 234 544 | **−368 268 ₽** |
| без 429 | конвенция курса | **235 876** | **0,00 ₽** |

Посмотрите на третью строку. Справочник **починен правильно** — и всё равно минус
триста шестьдесят восемь тысяч. Студент, который выбросил верную строку и увидел
такое, сделает вывод, что выбросил не ту. И будет неправ.

Недостача — это **1 332 платежа, и все до единого за 28 февраля 2026**.
Потому что `valid_to` — это `date`, то есть полночь, а платёж случился днём.
Буквальное `BETWEEN` выбрасывает весь последний день каждого интервала.

**Конвенция границы на весь курс — до декабря и в ДЗ-2:**

```python
(paid_at >= valid_from) & (paid_at < valid_to + pd.Timedelta(days=1))
```

**Понимать.** Дефект нашёлся легко, а выбор между двумя починками сделала не
интуиция, а число, посчитанное независимо. Вот зачем нужен контроль, который
не использует то, что проверяет.

> На всей базе те же четыре прогона дают 880 737 565,00 ₽ при разнице 0,00 —
> и 1 577 870 ₽ недобора на 5 730 платежах, тоже все за 28 февраля.


In [ ]:
priced = join_prices(pay_t, fixed, between=False)
diff = round(priced["amount"].sum() - control, 2)

protocol.append({"step": "цена по справочнику", "rows_before": len(pay_t),
                 "rows_after": len(priced), "validate": "интервал",
                 "keys_lost": len(pay_t) - len(priced),
                 "verdict": "ok" if diff == 0 else f"ВНИМАНИЕ: разница {diff}"})

print("разница с контролем:", diff)


---

## 3. Шеринг: ошибка сразу в две стороны

Подпиской ПРАЙМ пользуется не только владелец: он подключает к ней до пяти
человек. Эти люди лежат в `subscription_members`, и витрине они нужны — иначе мы
посчитаем, что экосистемой пользуется вдвое меньше народу, чем на самом деле.

Присоединить их наивно нельзя по той же причине, что и платежи. Но здесь есть
и вторая ловушка, потоньше.


In [ ]:
members = q(f"""
    select m.subscription_id, m.client_id, m.joined_at
    from prime.subscription_members m
    join prime.clients c on c.client_id = m.client_id
    where c.{SEGMENT}
    order by m.client_id, m.subscription_id
""")

print(len(members), "строк участия")
members.head(3)


In [ ]:
# Сколько клиентов сегмента не участвуют вообще ни в одной подписке?
# Назовите число ДО запуска — и мы сверим.
probe = clients.merge(members[["client_id"]].drop_duplicates(),
                      on="client_id", how="left", indicator=ЗАПОЛНИТЕ)

print(probe["_merge"].value_counts())


**37 037 клиентов из 112 415 не участвуют ни в одной подписке.** Треть сегмента —
и это **не ошибка**: участник — это тот, с кем поделились, а сам владелец в этой
таблице не числится вовсе. Человек может владеть подпиской и не быть ничьим
участником.

Ошибкой это стало бы, если бы мы **не ждали** такого числа. Поэтому `indicator=True`
и ставят: он отвечает не на вопрос «сколько строк получилось», а на вопрос
«**какие именно** строки не нашли пару» — и их можно посмотреть глазами.

**Уметь писать.** `indicator=True` на каждом левом соединении, где вы не уверены,
что пара найдётся всем.


In [ ]:
# Сворачиваем участия перед соединением. До чего сворачивать — до подписки
# или до клиента? Что получится в строке ЛЮДИ, если ошибиться?
memberships = (members
               .groupby(ЗАПОЛНИТЕ, as_index=False)
               .agg(n_memberships=("subscription_id", "size")))

print(len(memberships), "строк после свёртки")


In [ ]:
mart_before = mart
mart = mart.merge(memberships, on="client_id", how="left", validate="1:1")

# Строку протокола пишем ДО fillna: после него пропусков уже не видно.
protocol.append(step("участия -> клиент", mart_before, mart, "n_memberships", "1:1", N - len(memberships)))

mart["n_memberships"] = mart["n_memberships"].fillna(0).astype("int64")
mart["is_owner"] = mart["subscription_id"].notna()
mart["uses_prime"] = mart["is_owner"] | (mart["n_memberships"] > 0)

print("владельцев:          ", int(mart["is_owner"].sum()))
print("участников:          ", int((mart["n_memberships"] > 0).sum()))
print("пользуются всего:    ", int(mart["uses_prime"].sum()))


**Свернуть до клиента, а не до подписки.** Если свернуть до подписки, то в витрину
приедет число **мест** — сколько раз кого-то куда-то подключили. Один человек
с тремя подписками даст три места, и строка ЛЮДИ насчитает людей больше, чем их
есть.

На нашем сегменте это **124 652 места против 100 202 людей**, на всей базе разрыв
вдвое: **889 763 места против 445 620 людей.** Одна буква в `groupby` — и это
разные сущности, а не два способа посчитать одно и то же.

Через минуту мы сломаем сборку именно здесь и посмотрим, как протокол это ловит.


---

## 4. Протокол сборки

Всё, что мы делали, надо теперь **доказать** — и не словами, а печатной таблицей.
Она называется протоколом сборки, и её форма фиксирована.

Три последние строки — это три инварианта витрины:

- **ЗЕРНО** — строк в витрине столько же, сколько уникальных клиентов, и столько,
  сколько мы объявили до сборки;
- **ДЕНЬГИ** — сумма по витрине минус независимый контроль равна нулю;
- **ЛЮДИ** — число тех, кто пользуется экосистемой, сходится со счётом, сделанным
  другим путём.

> **В ДЗ-2 протокол стоит 3 балла из 10.** Без него ноль, даже если все числа
> верные: результат, который нельзя перепроверить, в работе не принимают.


In [ ]:
control_people = q(f"""
    select count(*) as people from prime.clients c
    where c.{SEGMENT}
      and (exists (select 1 from prime.subscriptions s where s.owner_client_id = c.client_id)
        or exists (select 1 from prime.subscription_members m where m.client_id = c.client_id))
""").iloc[0, 0]

grain_ok = len(mart) == mart["client_id"].nunique() == N
money_diff = round(mart["paid"].fillna(0).sum() - control, 2)
people = int(mart["uses_prime"].sum())

print(pd.DataFrame(protocol).to_string(index=False))
print()
print(f"ЗЕРНО   строк {len(mart)}, клиентов {mart['client_id'].nunique()}, объявляли {N}"
      f"   -> {'ok' if grain_ok else 'ВНИМАНИЕ'}")
print(f"ДЕНЬГИ  разница с независимым контролем: {money_diff:.2f} ₽"
      f"   -> {'ok' if money_diff == 0 else 'ВНИМАНИЕ'}")
print(f"ЛЮДИ    по витрине {people}, независимо {control_people}"
      f"   -> {'ok' if people == control_people else 'ВНИМАНИЕ'}")


In [ ]:
# Ломаем намеренно: забываем свернуть участия и соединяем как есть.
broken = clients.merge(members, on="client_id", how="left")
seats = int(broken["subscription_id"].notna().sum())

print(f"строк: {len(broken)}, а клиентов {N}")
print(f"ЗЕРНО   -> {'ok' if len(broken) == N else 'ВНИМАНИЕ: одна строка больше не один клиент'}")
print(f"ЛЮДИ    -> такая таблица даёт {seats} МЕСТ вместо {control_people} ЛЮДЕЙ")


**Проверка, которая не может упасть, — не проверка.**

Мы только что сломали одну строку сборки, и протокол покраснел в двух местах.
Если бы он не покраснел — он ничего и не проверял бы, а просто печатал красивые
числа рядом друг с другом.

Отсюда правило, по которому протокол пишут: **сравнивать надо не с нулём,
а с ожиданием, объявленным заранее.** «Потеряли 37 232 клиента» — это `ok`, если
вы до сборки сказали, что у стольких нет подписки. И это `ВНИМАНИЕ`, если вы этого
не говорили.


---

## Через три дня

В субботу выйдет **ДЗ-2**: та же витрина, но на вашем личном сегменте и с пятью
показателями поверх неё. Протокол сборки там — обязательная часть, той же формы,
что сегодня.

Шаги 1 и 2 из хвоста ниже — это ровно то, с чего вы начнёте. Остальное — новое.

Текст задания появится 26 сентября; сегодня искать его не надо.


---

## Что унести с сегодняшнего дня

1. **Зерно объявляют словами и до кода.** «Одна строка — один клиент» — это
   утверждение, которое потом проверяется числом.
2. **Сворачивай до соединения, а не после.** `groupby` + `agg` отдельной таблицей,
   и только потом `merge`.
3. **Кратность объявляют:** `validate="1:1"` падает ровно тогда, когда ошиблись вы.
4. **`validate` проверяет ключ, а не интервал.** Справочник с историей цен он
   не спасёт — там работает самосоединение.
5. **Граница интервала:** `>= valid_from` и `< valid_to + 1 день`. Буквальное
   `BETWEEN` тихо теряет последний день.
6. **Контрольная сумма выбирает между двумя правдоподобными починками.** Глаза
   не выбирают.
7. **Протокол, который не может покраснеть, ничего не доказывает.**

### Где вы запустили то, что было на лекции

| Слайд Л4 | Приём | Где |
|---|---|---|
| 4–5 | множитель считается до соединения | раздел 1, наивная сборка |
| 7 | `how=` и судьба непарных строк | раздел 1 |
| 8 | `validate=` объявляет кратность | раздел 1 |
| 9 | `indicator=True` | раздел 3 |
| 10 | свернуть сторону «многие» до зерна | разделы 1 и 3 |
| 13–14 | справочник с историей, самосоединение, граница интервала | раздел 2 — ядро |
| 15 | протокол сборки | раздел 4 |
| 17–18 | `agg`, сумма по группам не сходится с общей | раздел 1 |
| 11 | `concat` вниз и три ловушки | ★1 |
| 19–21 | `transform` и `filter` | ★2 |
| 23–24 | `pivot_table` и `melt` | ★3 |
| 25 | топ-N «вообще» и «в каждой группе» | ★4 |
| 26 | распространение форм, `np.where` против `apply` | ★5 |
| 28 | шесть таблиц витрины | весь семинар; шестая — задача 4 хвоста |

### Сроки

**ДЗ-1 — 27 сентября, 23:59.** Приём со снижением балла до 4 октября.
**ДЗ-2** выдаётся 26 сентября, дедлайн 3 октября.


In [ ]:
ANSWERS = {
    "★3-хвост": "7f4ea2c484",
    "★1": "6ebe382078",
    "★2": "bf96401cc2",
    "★3": "83c2579b01",
    "★4": "a90fe35ba5",
    "★5": "06df7c0c54"
}


In [ ]:
import hashlib

import numpy as np
import pandas as pd


def _norm(x):
    if isinstance(x, bool):
        return repr(x)
    if isinstance(x, float):
        return f"{x:.2f}"
    if isinstance(x, int):
        return str(x)
    if isinstance(x, dict):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, (set, frozenset)):
        return "{" + ",".join(sorted(_norm(v) for v in x)) + "}"
    if isinstance(x, (list, tuple)):
        return "[" + ",".join(_norm(v) for v in x) + "]"
    if isinstance(x, np.integer):
        return str(int(x))
    if isinstance(x, np.bool_):
        return repr(bool(x))
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    if isinstance(x, pd.Timedelta):
        return str(int(x.total_seconds()))
    if isinstance(x, pd.Index):
        return _norm(list(x))
    if isinstance(x, pd.Series):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, pd.DataFrame):
        d = x if isinstance(x.index, pd.RangeIndex) else x.reset_index()
        cols = sorted(map(str, d.columns))
        rows = sorted(tuple(_norm(r[c]) for c in cols) for _, r in d.iterrows())
        return "[" + ",".join("[" + ",".join(r) + "]" for r in rows) + "]"
    return repr(x)


def _show(x) -> str:
    """Короткий показ ответа: кадр на сто тысяч строк в вывод не печатаем."""
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return f"{type(x).__name__} {x.shape}\n{x.head().to_string()}"
    return repr(x)


def check(task: str, got) -> None:
    """Сверить ответ с эталоном. Эталоны лежат хешами — подсмотреть нельзя."""
    if got is None:
        print(f"{task}: не решено")
        return
    digest = hashlib.sha256(_norm(got).encode()).hexdigest()[:10]
    if not ANSWERS:
        print(f"HASH:{task}={digest}")
    elif digest == ANSWERS.get(task):
        print(f"{task}: верно")
    else:
        print(f"{task}: не сходится, получилось {_show(got)}")


print("проверка готова")


---

## Хвост: ваш сегмент

Дальше — работа с открытым концом. Сколько успеете на паре, столько успеете;
остальное дома. Ничего из этого не сдаётся.

Задачи 1 и 2 — это то, с чего начнётся ДЗ-2, только там сегмент будет ваш личный
из `prime.assignments`, а не общий.


In [ ]:
# Задача 1. Свой сегмент и своё контрольное число.
# Возьмите таблицу и признак из своей строки prime.assignments.
mine = q("""
    select client_id, region_code, acquisition_channel
    from prime.clients
    where ЗАПОЛНИТЕ
    order by client_id
""")

print("ЗЕРНО моего сегмента:", len(mine))


In [ ]:
# Задача 2. Соберите по своему сегменту витрину с двумя колонками:
# сколько платежей и сколько заплатил. Напечатайте протокол своей сборки.
my_mart = None

print(my_mart if my_mart is None else len(my_mart))


### ★ Задача 3. На сколько врёт витрина без починки справочника

Сколько лишних рублей приедет в общий сегмент этого семинара, если соединить
платежи с **непочиненным** справочником по конвенции курса?

Считайте на общем сегменте занятия (`pay_t`, `ref`), а не на своём — иначе у всех
получится своё число, и проверка не сработает.

**Ответ:** одно число, рубли с двумя знаками — `round(extra, 2)`.


In [ ]:
answer = None   # ← ваш расчёт

check("★3-хвост", answer)


### Задача 4. Дно

Посчитайте CAC по каналам привлечения: сколько стоило привести одного клиента.
Расходы лежат в `prime.marketing_spend`, зерно там — «канал × день».

Эта задача намеренно трудная и за пару не доделывается. В ней спрятан главный
ответ темы: **не всё, что нужно витрине, к витрине присоединяется.**
`marketing_spend` не имеет `client_id` — и никогда не будет иметь.


In [ ]:
# Задача 4. CAC по каналам. Соединять marketing_spend с витриной НЕ нужно —
# подумайте, почему, и что с этим делать.
cac = None

print(cac)


---

## Задачи со звёздочкой

Пять ловушек с лекции, которые мы сегодня не трогали руками. **Не сдаются
и не оцениваются**, но каждая встретится в проекте.

Проверка — та же, что в хвосте: она знает ответ, но не показывает его. Формат
ответа задан в условии, без него хеш не сойдётся.


### ★ 1. `concat` соединяет вниз — и три раза подставляет

Слайд 11. Разрежьте платежи на два куска по дате и во втором переименуйте колонку
`amount` в `amount_rub`. Склейте `pd.concat` и посчитайте сумму по `amount`.

**Ответ:** список `[сумма после наивного concat, сумма после правильного]`,
оба числа с двумя знаками.


In [ ]:
first = pay[pay["paid_at"] < "2026-04-01"]
second = pay[pay["paid_at"] >= "2026-04-01"].rename(columns={"amount": "amount_rub"})

answer = None   # ← ваш расчёт

check("★1", answer)


### ★ 2. Группа проходит целиком или не проходит вовсе

Слайды 19–21. Оставьте только подписки, у которых **шесть и больше** платежей
за окно. Сделайте это двумя способами: через `groupby().filter()` и через
`transform("size")` с маской.

**Ответ:** список `[строк после filter, строк после transform]`. Числа обязаны
совпасть — если нет, один из способов написан неверно.


In [ ]:
answer = None   # ← ваш расчёт

check("★2", answer)


### ★ 3. Широкая для человека, длинная для кода

Слайды 23–24. Постройте сводную таблицу «месяц × тариф» с суммой платежей:
месяц берётся из `paid_at`, тариф — из `pay_t`. Потом разверните её обратно
в длинный формат через `melt`.

Про `to_period` на колонке с часовым поясом pandas ругается предупреждением —
уберите зону через `.dt.tz_localize(None)` перед этим.

**Ответ:** список `[строк в широкой, колонок в широкой, строк в длинной]`.


In [ ]:
answer = None   # ← ваш расчёт

check("★3", answer)


### ★ 4. Топ «вообще» и топ «в каждом»

Слайд 25. Сложите выручку по паре «канал привлечения × тариф» (канал берите
из витрины, тариф — из `pay_t`). Верните **три самые большие суммы** по всей
таблице, по убыванию.

**Ответ:** список из трёх чисел с двумя знаками.


In [ ]:
answer = None   # ← ваш расчёт

check("★4", answer)


### ★ 5. Распространение форм

Слайд 26. Пометьте платежи дороже 300 ₽ словом `"крупный"`, остальные — `"обычный"`,
двумя способами: `np.where` и `.apply(axis=1)`. Замерьте оба через `%%timeit`
и посмотрите на разницу — она будет примерно стократной.

Обратите внимание, что `np.where` получает **одно** значение `"крупный"`, а
возвращает колонку на двести тысяч строк: это и есть распространение форм.

**Ответ:** одно целое число — сколько платежей оказались крупными.


In [ ]:
import numpy as np

answer = None   # ← ваш расчёт

check("★5", answer)
